In [ ]:
# Make sure this GitHub repo contains code/evaluate_cross_dataset.py before running on Kaggle.
# If you upload the edited repo as a Kaggle Dataset instead, replace this cell with a cp -r command.
REPO_URL = "https://github.com/DinhLiu/Generalizable-FER.git"
!rm -rf /kaggle/working/Generalizable-FER
!git clone {REPO_URL} /kaggle/working/Generalizable-FER

In [ ]:
%cd /kaggle/working/Generalizable-FER

In [ ]:
%%capture
!pip install -r requirements.txt

## Kaggle input structure

This notebook expects the Kaggle inputs from `main.pdf`:

- FERPlus: `test/train/validation` with class folders, including `suprise` and `contempt`.
- AffectNet: `archive (3)/Test` and `archive (3)/Train` with class folders.
- SFEW2.0: `Train` and `Val` with class folders; `Test/Test_Aligned_Faces` is unlabeled, so evaluation uses `Val`.
- MMAFEDB: `MMAFEDB/test`, `MMAFEDB/train`, and `MMAFEDB/valid` with class folders.

`contempt` is skipped because the CAFE checkpoint trained on RAF-DB has 7 outputs and no contempt class.

In [ ]:
from pathlib import Path
import os
import shutil

WORK_DATA = Path("/kaggle/working/data")
WORK_DATA.mkdir(parents=True, exist_ok=True)
Path("/kaggle/working/outputs").mkdir(parents=True, exist_ok=True)

LINKS = {
    "raf-basic": [
        "/kaggle/input/raf-db-dataset",
        "/kaggle/input/datasets/shuvoalok/raf-db-dataset",
        "/kaggle/input/raf-db/raf-basic",
    ],
    "ckplus48": [
        "/kaggle/input/ckdataset",
        "/kaggle/input/ckplus48",
        "/kaggle/input/datasets/davilsena/ckdataset",
    ],
    "ferplus": [
        "/kaggle/input/ferplus",
        "/kaggle/input/datasets/arnabkumarroy02/ferplus",
    ],
    "affectnet": [
        "/kaggle/input/affectnet",
        "/kaggle/input/datasets/mstjebashazida/affectnet",
    ],
    "sfew": [
        "/kaggle/input/datasetsfew",
        "/kaggle/input/dataset-sfew",
        "/kaggle/input/datasets/vlntnstarodub/datasetsfew",
    ],
    "mma": [
        "/kaggle/input/mma-facial-expression",
        "/kaggle/input/datasets/mahmoudima/mma-facial-expression",
    ],
    "resnet18_msceleb.pth": [
        "/kaggle/input/resnet18-msceleb/resnet18_msceleb.pth",
        "/kaggle/input/datasets/vnnhhiu/resnet18-msceleb/resnet18_msceleb.pth",
    ],
}
OPTIONAL = {"ckplus48"}


def first_existing(candidates):
    for candidate in candidates:
        p = Path(candidate)
        if p.exists():
            return p
    return None


def replace_link(dst, src):
    dst = Path(dst)
    if dst.is_symlink() or dst.is_file():
        dst.unlink()
    elif dst.exists():
        shutil.rmtree(dst)
    os.symlink(src, dst, target_is_directory=src.is_dir())

missing = []
for name, candidates in LINKS.items():
    src = first_existing(candidates)
    if src is None:
        if name not in OPTIONAL:
            missing.append((name, candidates))
        print(f"SKIP {name}: not found")
        continue
    dst = Path("/kaggle/working") / name if name.endswith(".pth") else WORK_DATA / name
    replace_link(dst, src)
    print(f"{name}: {src} -> {dst}")

if missing:
    msg = "\n".join(f"{name}: {candidates}" for name, candidates in missing)
    raise FileNotFoundError(f"Missing required Kaggle inputs:\n{msg}")

In [ ]:
from pathlib import Path

for root in [
    Path("/kaggle/working/data/ferplus"),
    Path("/kaggle/working/data/affectnet"),
    Path("/kaggle/working/data/sfew"),
    Path("/kaggle/working/data/mma"),
]:
    print(f"\n== {root} ==")
    for path in sorted(p for p in root.rglob("*") if p.is_dir()):
        rel = path.relative_to(root)
        if len(rel.parts) <= 3:
            print(rel)

In [ ]:
%cd /kaggle/working/Generalizable-FER

!python scripts/create_raf_compatible.py \
  --src /kaggle/working/data/raf-basic \
  --dst /kaggle/working/data/raf-basic-compatible

In [ ]:
# %cd /kaggle/working/Generalizable-FER/code
# Smoke test:
# !python ours_CAFE.py \
#   --raf_path ../../data/raf-basic-compatible \
#   --label_path list_patition_label.txt \
#   --batch_size 32 \
#   --workers 2 \
#   --gpu 0 \
#   --epochs 1

In [ ]:
%cd /kaggle/working/Generalizable-FER/code
!python ours_CAFE.py \
  --raf_path ../../data/raf-basic-compatible \
  --label_path list_patition_label.txt \
  --batch_size 32 \
  --workers 2 \
  --gpu 0 \
  --epochs 60

In [ ]:
!mkdir -p /kaggle/working/outputs/rafdb_cafe_seed3407
!cp ours_best.pth ours_final.pth results.txt /kaggle/working/outputs/rafdb_cafe_seed3407/

In [ ]:
!zip -r /kaggle/working/outputs/rafdb_cafe_seed3407.zip /kaggle/working/outputs/rafdb_cafe_seed3407

In [ ]:
import subprocess
from pathlib import Path

CODE_DIR = Path("/kaggle/working/Generalizable-FER/code")
CHECKPOINT = "/kaggle/working/outputs/rafdb_cafe_seed3407/ours_best.pth"
ck_path = Path("/kaggle/working/data/ckplus48")

if ck_path.exists():
    subprocess.run([
        "python", "evaluate_ckplus48.py",
        "--ck_path", str(ck_path),
        "--checkpoint", CHECKPOINT,
        "--output_dir", "/kaggle/working/outputs/ckplus48_eval",
        "--batch_size", "64",
        "--workers", "2",
        "--gpu", "0",
    ], cwd=CODE_DIR, check=True)
    subprocess.run([
        "python", "evaluate_ckplus48.py",
        "--ck_path", str(ck_path),
        "--checkpoint", CHECKPOINT,
        "--output_dir", "/kaggle/working/outputs/ckplus48_peak_eval",
        "--batch_size", "64",
        "--workers", "2",
        "--gpu", "0",
        "--exclude_neutral",
    ], cwd=CODE_DIR, check=True)
else:
    print("CK+48 input is not linked; skipping CK+48 evaluation.")

In [ ]:
import subprocess
from pathlib import Path

CODE_DIR = Path("/kaggle/working/Generalizable-FER/code")
CHECKPOINT = "/kaggle/working/outputs/rafdb_cafe_seed3407/ours_best.pth"

PAPER_EVALS = [
    ("ferplus", "/kaggle/working/data/ferplus", None, "/kaggle/working/outputs/ferplus_eval"),
    ("affectnet", "/kaggle/working/data/affectnet", None, "/kaggle/working/outputs/affectnet_eval"),
    ("sfew", "/kaggle/working/data/sfew", "Val", "/kaggle/working/outputs/sfew_val_eval"),
    ("mma", "/kaggle/working/data/mma", None, "/kaggle/working/outputs/mma_eval"),
]

for dataset, data_path, split, output_dir in PAPER_EVALS:
    cmd = [
        "python", "evaluate_cross_dataset.py",
        "--dataset", dataset,
        "--data_path", data_path,
        "--checkpoint", CHECKPOINT,
        "--output_dir", output_dir,
        "--batch_size", "64",
        "--workers", "2",
        "--gpu", "0",
    ]
    if split:
        cmd.extend(["--split", split])
    print("\nRunning", dataset, "...")
    subprocess.run(cmd, cwd=CODE_DIR, check=True)

In [ ]:
import json
from pathlib import Path
import pandas as pd

rows = []
for metrics_path in sorted(Path("/kaggle/working/outputs").glob("*_eval/metrics.json")):
    metrics = json.loads(metrics_path.read_text())
    rows.append({
        "dataset": metrics.get("dataset", metrics.get("dataset_mode", metrics_path.parent.name)),
        "dataset_mode": metrics.get("dataset_mode"),
        "num_samples": metrics.get("num_samples"),
        "overall_accuracy": metrics.get("overall_accuracy"),
        "mean_accuracy": metrics.get("mean_accuracy"),
        "macro_f1": metrics.get("macro_f1"),
        "skipped_classes": metrics.get("skipped_classes", {}),
        "output_dir": str(metrics_path.parent),
    })

summary = pd.DataFrame(rows).sort_values("dataset_mode")
summary.to_csv("/kaggle/working/outputs/cross_dataset_summary.csv", index=False)
display(summary)

In [ ]:
!cd /kaggle/working && zip -r outputs/cross_dataset_evaluations.zip outputs/*_eval outputs/cross_dataset_summary.csv